# 04 — Call Summarization

**Goal:** for every processed call, a short summary of the **customer issue**, the **agent action**, and the **outcome or next step**.

**Runs on:** Google Colab with a GPU (Runtime → Change runtime type → T4 GPU).

**Input:** `results/segmented_turns.csv` — our own Whisper transcripts with speaker roles (notebook 03). The human reference transcripts are **not** used here.

**Output**
- `results/summaries.csv` — one summary per call, with automatic check results
- `results/summary_review.csv` — 10 calls prepared for manual review
- `results/summarization_run_info.json` — model and library versions

**Sections**
1. Choosing a summarization approach
2. Setup
3. Load the language model
4. The prompt
5. Generate the summaries
6. Automatic checks
7. Manual review sheet
8. Findings
9. Save and download (Colab)

## 1. Choosing a summarization approach

| Approach | How it works | Fit for this task |
|---|---|---|
| **Extractive** | Picks existing sentences from the call | Poor: no single sentence states issue + action + outcome; spoken sentences are fragmented |
| **Rule-based** | Templates and keywords (for example "transfer" → issue = transfer) | Works only for the 8 known intents; cannot describe what actually happened |
| **Abstractive (model-based)** | A language model writes new sentences | Good fit: flexible and readable. **Risk:** it can invent details (hallucination) |

We use **abstractive summarization with a small open-source instruction-tuned model** (`Qwen3-4B-Instruct-2507`, Apache-2.0 license; fallback `Qwen2.5-1.5B-Instruct`). It runs on a free Colab GPU, needs no paid API, and no data leaves the notebook.

**How we limit hallucination**
1. The prompt says: use only what is stated; write *"not stated"* otherwise.
2. **Greedy decoding** (no random sampling): the same transcript always gives the same summary.
3. A fixed **JSON format** with three fields, so every summary has the same structure.
4. **Automatic checks** (section 6) and a **manual review** of 10 calls (section 7).

## 2. Setup

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ahmedalmoutiry-wq/call-center-agent-performance-analysis.git"
REPO_DIR = "/content/call-center-agent-performance-analysis"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
    !pip install -q num2words==0.5.14

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import json
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import RESULTS_DIR
from src.llm import DEFAULT_MODELS, load_llm
from src.summarize import (NOT_STATED, SUMMARY_FIELDS, build_messages, issue_matches_task,
                           parse_summary, transcript_text, ungrounded_numbers)

SEED = 42
pd.set_option("display.max_colwidth", 120)

turns = pd.read_csv(RESULTS_DIR / "segmented_turns.csv")
subset = pd.read_csv(RESULTS_DIR / "subset_ids.csv")
print(f"Calls: {turns['call_id'].nunique()}  |  turns: {len(turns)}")

Calls: 25  |  turns: 237


## 3. Load the language model

In [3]:
generate, MODEL_ID = load_llm(DEFAULT_MODELS)

import torch, transformers
run_info = {"model": MODEL_ID, "transformers": transformers.__version__, "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
            "decoding": "greedy (do_sample=False)", "max_new_tokens": 200}
run_info

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

{'model': 'Qwen/Qwen3-4B-Instruct-2507',
 'transformers': '5.21.0',
 'torch': '2.11.0+cu130',
 'gpu': 'Tesla T4',
 'decoding': 'greedy (do_sample=False)',
 'max_new_tokens': 200}

In [12]:
!pip list 2>/dev/null | grep -iE "^(accelerate|transformers|torch|safetensors) "

accelerate                            1.15.0
safetensors                           0.8.0
torch                                 2.11.0+cu130
transformers                          5.21.0


## 4. The prompt

The **system message** sets the rules (only stated facts, "not stated" otherwise). The **user message** contains the transcript and the output format. Here is the full prompt for one call:

In [4]:
example_call = turns["call_id"].iloc[0]
for message in build_messages(turns[turns["call_id"] == example_call]):
    print(f"--- {message['role'].upper()} ---")
    print(message["content"])
    print()

--- SYSTEM ---
You summarize customer-service phone calls for a quality team. Use ONLY information that is stated in the transcript. Never guess or add details. If something is not stated in the call, write "not stated".

--- USER ---
Transcript of a call between a bank Agent and a Customer:

Agent: Hello, this is Harbor Valley National Bank. My name is Robert. How can I help you today?
Customer: Hi. My name is Linda Smith. I need to check my account balance.
Agent: Sure, I'd be happy to do that for you. And what's your name, please? Oh, right, it's Linda Smith. Sorry about that. Linda Smith, right.
Customer: Linda Smith.
Agent: Alright, I'm in. And which account would you like to check? Your savings account or your checking account?
Customer: All is well. I'd like to check my checking account, please.
Agent: Your checking account balance is $90 exactly.
Customer: Fantastic. That's all I needed.
Agent: Great. Is there anything else I could help you with today?
Customer: Nope. That'll b

## 5. Generate the summaries

In [5]:
rows, started = [], time.perf_counter()
for call_id, call_turns in turns.groupby("call_id"):
    raw = generate(build_messages(call_turns))
    rows.append({"call_id": call_id, **parse_summary(raw), "raw_output": raw})

summaries = pd.DataFrame(rows)
print(f"Generated {len(summaries)} summaries in {time.perf_counter() - started:.0f} s")
summaries[["call_id"] + SUMMARY_FIELDS].head(5)

Generated 25 summaries in 118 s


,call_id,customer_issue,agent_action,outcome
0,08f360b3f3b844ef,The customer wanted to check her checking account balance.,The agent confirmed the customer's identity and provided her checking account balance of $90.,The customer expressed satisfaction and stated there was nothing else needed.
1,09fc75fc02ea4b46,The customer needed a new checkbook.,The agent confirmed the address and sent a new checkbook to the customer's home address.,The call ended with the customer stating there was no further need for assistance.
2,0dee81761a5c45f9,The customer wanted to check the balance of his savings account.,The agent confirmed and provided the savings account balance of $46.,"The customer stated there was no other need, and the call ended."
3,0f87dc392db84250,The customer needed a new checkbook.,The agent confirmed the new checkbook was sent to the customer's address.,The call ended with the customer thanking the agent and saying goodbye.
4,37a7a35a1e244820,The customer wanted to know the local branch hours.,The agent provided the branch hours: 9:30 a.m. to 5 p.m.,The call ended with the customer thanking the agent and saying goodbye.


## 6. Automatic checks

These checks need no human labels. They cannot prove a summary is good, but they catch clear problems:

| Check | What it catches |
|---|---|
| **Format** — all three fields present | The model did not follow the JSON format |
| **Issue matches the known task** — the issue mentions the call's task type (for example "transfer") | A wrong customer issue. The task type comes from the dataset metadata, which we did not give the model |
| **Numbers are grounded** — every amount, time or number in the summary appears in the transcript | Invented details (hallucination) |
| **"not stated" count** | How often the model admits information is missing |

In [6]:
summaries = summaries.merge(subset[["call_id", "task_type"]], on="call_id")
transcripts = {c: transcript_text(t) for c, t in turns.groupby("call_id")}

summaries["issue_matches_task"] = [issue_matches_task(i, t)
                                   for i, t in zip(summaries["customer_issue"], summaries["task_type"])]
summaries["ungrounded_numbers"] = [
    ", ".join(ungrounded_numbers(" ".join(r[f] for f in SUMMARY_FIELDS), transcripts[r["call_id"]]))
    for _, r in summaries.iterrows()]

checks = pd.Series({
    "summaries": len(summaries),
    "format OK": int(summaries["parse_ok"].sum()),
    "issue matches the known task": int(summaries["issue_matches_task"].sum()),
    "summaries with ungrounded numbers": int((summaries["ungrounded_numbers"] != "").sum()),
    **{f'"{NOT_STATED}" in {f}': int(summaries[f].str.lower().str.contains(NOT_STATED).sum()) for f in SUMMARY_FIELDS},
})
checks.to_frame("count")

,count
summaries,25
format OK,25
issue matches the known task,25
summaries with ungrounded numbers,0
"""not stated"" in customer_issue",0
"""not stated"" in agent_action",0
"""not stated"" in outcome",0


In [7]:
# Calls that failed a check, to read them closely
problems = summaries[~summaries["parse_ok"] | ~summaries["issue_matches_task"] | (summaries["ungrounded_numbers"] != "")]
problems[["call_id", "task_type", "customer_issue", "outcome", "ungrounded_numbers"]]

,call_id,task_type,customer_issue,outcome,ungrounded_numbers


In [8]:
# Read one flagged call next to its transcript (if any call was flagged)
if len(problems):
    call_id = problems["call_id"].iloc[0]
    print(transcripts[call_id], "\n")
    print(summaries.loc[summaries["call_id"] == call_id, SUMMARY_FIELDS].T.to_string())

## 7. Manual review sheet

Automatic checks cannot judge whether a summary is **faithful** (says nothing false) and **complete** (covers issue, action and outcome). A person must read them.

We pick 10 calls at random (fixed seed), always including the edge-case call. The sheet `results/summary_review.csv` has empty columns to fill in after reading each transcript and summary:
- `faithful` — `yes` / `no`: is every statement supported by the transcript?
- `complete` — `yes` / `no`: are issue, action and outcome all correct (or correctly "not stated")?
- `review_notes` — what is wrong, if anything

In [9]:
edge_call = subset.loc[subset["agent_on_channel_1"], "call_id"].tolist()
others = summaries.loc[~summaries["call_id"].isin(edge_call), "call_id"].sample(n=10 - len(edge_call), random_state=SEED)
review_ids = edge_call + others.tolist()

review = summaries[summaries["call_id"].isin(review_ids)][["call_id", "task_type"] + SUMMARY_FIELDS].copy()
review.insert(2, "transcript", review["call_id"].map(transcripts))
review["faithful"], review["complete"], review["review_notes"] = "", "", ""
review.to_csv(RESULTS_DIR / "summary_review.csv", index=False)
print(f"Review sheet: {len(review)} calls -> results/summary_review.csv")

Review sheet: 10 calls -> results/summary_review.csv


In [10]:
# Print the 10 calls to read them here
for r in review.itertuples():
    print(f"===== {r.call_id}  ({r.task_type}) =====")
    print(r.transcript)
    print(f"\n  ISSUE:   {r.customer_issue}\n  ACTION:  {r.agent_action}\n  OUTCOME: {r.outcome}\n")

===== 08f360b3f3b844ef  (check balance) =====
Agent: Hello, this is Harbor Valley National Bank. My name is Robert. How can I help you today?
Customer: Hi. My name is Linda Smith. I need to check my account balance.
Agent: Sure, I'd be happy to do that for you. And what's your name, please? Oh, right, it's Linda Smith. Sorry about that. Linda Smith, right.
Customer: Linda Smith.
Agent: Alright, I'm in. And which account would you like to check? Your savings account or your checking account?
Customer: All is well. I'd like to check my checking account, please.
Agent: Your checking account balance is $90 exactly.
Customer: Fantastic. That's all I needed.
Agent: Great. Is there anything else I could help you with today?
Customer: Nope. That'll be everything. Thank you very much.
Agent: Thank you for calling, have a great day.
Customer: useful

  ISSUE:   The customer wanted to check her checking account balance.
  ACTION:  The agent confirmed the customer's identity and provided her check

## 8. Findings

Setup

Model: Qwen/Qwen3-4B-Instruct-2507 (Apache-2.0), transformers 5.21.0, Tesla T4, greedy decoding.
Input: our own Whisper transcripts with speaker roles (notebook 03), not the human reference.

Automatic checks (25 calls)

Format OK: 25/25. Issue matches the known task type: 25/25. Summaries with numbers not found in the transcript: 0/25.
The model never wrote "not stated", even when information was missing (see below).

Manual review (10 calls)

Faithful: 8/10 (95% CI 49%–94%). Complete: 9/10 (95% CI 60%–98%).
9e05bd9ca9564089: the recording ends after the customer's question, but the summary says the agent gave the branch hours and the customer received them. The model invented the usual ending of the script (hallucination).
663201e33f1a4e38: the outcome says the customer thanked the agent; only the agent said thank you.
Both errors are in the outcome field: the model tends to complete the call the way such calls usually end.

What would reduce these errors

Detect incomplete calls automatically (for example, no agent turn after the customer's request) and force "not stated".
Ask the model to quote the transcript for each field, then check the quotes, as done for sentiment in notebook 05.

Limitations

The automatic checks test format, the issue keyword and numbers only; the agent action and the outcome were checked on 10 calls by hand.
10 reviewed calls give wide confidence intervals.

## 9. Save and download (Colab)

In [11]:
out_cols = ["call_id", "task_type"] + SUMMARY_FIELDS + ["parse_ok", "issue_matches_task", "ungrounded_numbers", "raw_output"]
summaries.assign(model=MODEL_ID)[out_cols + ["model"]].to_csv(RESULTS_DIR / "summaries.csv", index=False)
(RESULTS_DIR / "summarization_run_info.json").write_text(json.dumps(run_info, indent=2))

if IN_COLAB:
    import shutil
    from google.colab import files
    export_dir = Path("/content/summary_results")
    export_dir.mkdir(exist_ok=True)
    for name in ["summaries.csv", "summary_review.csv", "summarization_run_info.json"]:
        shutil.copy(RESULTS_DIR / name, export_dir / name)
    shutil.make_archive("/content/summary_results", "zip", export_dir)
    files.download("/content/summary_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>